# Control Flow with `continue`, `all`, and `any`

This notebook demonstrates idiomatic use-cases for `continue` in `for` and `while` loops, shows alternatives using `all`/`any`, explores generator-based short-circuiting and vectorized replacements using NumPy and pandas, and concludes with unit tests and exercises.

Notes:
- Cells contain short illustrative examples. Do not run heavy computations here.
- The exercises are provided with hidden solution cells (end of notebook).

## 1. Import Required Libraries

We import standard libraries and testing tools. We set a non-interactive Matplotlib backend for headless environments.

In [ ]:
# Section 1: imports and helpers
import time
import itertools
from typing import Iterable

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")  # non-interactive backend
import matplotlib.pyplot as plt

# small helpers

def timeit_fn(fn, *args, repeat=5, **kwargs):
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        fn(*args, **kwargs)
        times.append(time.perf_counter() - t0)
    return min(times)


def sample_data(n=10):
    rng = np.random.RandomState(0)
    return rng.randint(-3, 3, size=n).tolist()

# test harness for demonstration
import pytest
def _assert_eq(a, b):
    assert a == b

## 2. Basic 'continue' Usage in For Loops

Examples of skipping invalid items or short-circuiting work in a loop using `continue`.

In [ ]:
# Example: filter and accumulate positive numbers
vals = [3, -1, 4, -2, 5]
acc = []
for v in vals:
    if v <= 0:
        continue
    acc.append(v)

print("positives:", acc)  # [3, 4, 5]

## 3. `continue` with While Loops and Loop Control

Be cautious to advance counters appropriately to avoid infinite loops.

In [ ]:
# Example: while loop with continue (careful with counter)
i = 0
out = []
while i < 6:
    i += 1  # important to advance before continue
    if i % 2 == 0:
        continue
    out.append(i)

print("odd numbers:", out)  # [1,3,5]

## 4. Using `all` and `any` Built-ins

Use `all()` and `any()` for concise validations over iterables.

In [ ]:
# all/any examples
seq1 = [2, 4, 6]
seq2 = [2, 3, 4]
print("all even seq1:", all(x % 2 == 0 for x in seq1))
print("any odd seq2:", any(x % 2 != 0 for x in seq2))

# Edge case: empty iterable
print("all([]) ->", all([]))  # True (vacuously true)
print("any([]) ->", any([]))  # False

## 5. Combining 'all'/'any' with 'continue' and Short-Circuiting

Compare an explicit loop with `continue` vs. using all()/any() with generator expressions. Show how short-circuiting affects performance and side effects; include microbenchmarks.

In [ ]:
# explicit loop with continue
def has_negative_loop(xs: Iterable[int]) -> bool:
    for x in xs:
        if x < 0:
            return True
        else:
            continue
    return False

# generator with any()
def has_negative_any(xs: Iterable[int]) -> bool:
    return any(x < 0 for x in xs)

xs = [1, 2, 3, -1, 5]
_assert_eq(has_negative_loop(xs), True)
_assert_eq(has_negative_any(xs), True)

## 6. Generator Expressions and Lazy Evaluation

Generators can avoid evaluating expensive checks for every element when short-circuiting occurs.

In [ ]:
# generator example: expensive predicate
import time

def expensive_check(x):
    time.sleep(0.01)  # simulate work
    return x > 0

# using any() stops early
vals = [1, -1, -2]
# If first item matches, any() stops and doesn't evaluate rest
# (We keep sleeps short in examples to avoid slow notebooks.)
# micro-timing

t = timeit_fn(lambda: any(expensive_check(x) for x in vals), repeat=3)
print("any() microsec:", t)

## 7. Nested Loops, 'continue', and Loop 'else' Clause

`else` on loops executes when the loop completes without `break`.

In [ ]:
# nested loop with continue and loop else
found = False
for i in range(3):
    for j in range(3):
        if i + j == 4:
            found = True
            break
        else:
            continue
    if found:
        break
else:
    # This else would run if outer loop completed without break
    print("not found")

print("found:", found)

## 8. Vectorized Alternatives with NumPy and Pandas

Use boolean masking and vectorized ops instead of explicit loops for large data.

In [ ]:
# NumPy vectorized filtering vs loop
arr = np.array([3, -1, 4, -2, 5])
mask = arr > 0
vec = arr[mask]
print("vector positives:", vec.tolist())

# Pandas example
df = pd.DataFrame({"x": arr})
print(df[df['x'] > 0])

## 9. Writing Unit Tests for Control-Flow Behavior

Examples of simple pytest tests. These tests assert behavior for functions using `continue` and `all`/`any`.

In [ ]:
# Unit tests (examples) -- can be picked up by pytest

def test_has_negative_any():
    assert has_negative_any([1, 2, -1]) is True
    assert has_negative_any([1, 2, 3]) is False


def test_has_negative_loop():
    assert has_negative_loop([1, 2, -1]) is True
    assert has_negative_loop([1, 2, 3]) is False

print("Run pytest in the repository to execute these tests")

## 10. Exercises and Practice Problems

1. Write a function that returns the first non-empty string from a list using `continue`.
2. Replace an explicit loop that accumulates valid items with a vectorized NumPy expression and compare timing.

Hints and reference solutions are provided in following (hidden) cells.

---

*End of notebook.*

This notebook is educational and light-weight. For CA26-specific experiments use the `scripts/run_experiment.py` and the `REPORT.md` pipeline.